In [9]:
import sys
from pathlib import Path

REPO_ROOT = Path(r"C:\Users\efthy\OneDrive - University College Dublin\power-systems-tools-learning")
PYTHON_ROOT = REPO_ROOT / "python"

if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))

In [17]:
from own_models.src.adapters.matpower_adapter import load_powercase_from_matpower_mat
from own_models.src.opf.dc.solve import solve_dc_opf_pyomo
case_path = REPO_ROOT / "data" / "raw" / "pglib" / "matpower_mat" / "base_case" / "pglib_opf_case14_ieee.mat"

case = load_powercase_from_matpower_mat(case_path)

print(case.name)
print("base_mva:", case.base_mva)
print("buses:", len(case.buses))
print("generators:", len(case.generators))
print("branches:", len(case.branches))
print("loads:", len(case.loads))
print("slack_bus_id:", case.slack_bus_id)

pglib_opf_case14_ieee
base_mva: 100.0
buses: 14
generators: 5
branches: 20
loads: 11
slack_bus_id: 1


In [18]:
active_branches = [b for b in case.branches if b.status == 1]

print("rated branches:", sum(b.rate_a > 0 for b in active_branches), "/", len(active_branches))
print("angle limited branches:", sum(True for b in active_branches if b.angmin <= b.angmax), "/", len(active_branches))
model, raw_results, result = solve_dc_opf_pyomo(
    case,
    solver_name="highs",
    tee=False,
    cost_power_units="mw",
    include_branch_limits=True,
    include_angle_limits=True,
)

rated branches: 20 / 20
angle limited branches: 20 / 20


In [19]:
print("status:", result.status)
print("termination:", result.termination_condition)
print("objective:", result.objective)
print("total_generation_pu:", result.total_generation)
print("total_load_pu:", result.total_load)
print("max_balance_residual:", result.max_power_balance_residual)

print("branch flow constraints:", len(model.branch_flow_limit))
print("angle constraints:", len(model.angle_difference_limit))

status: ok
termination: optimal
objective: 2051.526309000001
total_generation_pu: 2.5900000000000016
total_load_pu: 2.59
max_balance_residual: 3.3306690738754696e-15
branch flow constraints: 20
angle constraints: 20


In [20]:
import pandas as pd

pg_df = pd.DataFrame(
    [
        {
            "gen_id": gen_id,
            "pg_pu": pg,
            "pg_mw": pg * case.base_mva,
        }
        for gen_id, pg in result.pg.items()
    ]
)

pg_df

,gen_id,pg_pu,pg_mw
0,1,2.59,259.0
1,2,0.00,0.0
2,3,0.00,0.0
3,4,0.00,0.0
4,5,0.00,0.0


In [21]:
theta_df = pd.DataFrame(
    [
        {
            "bus_id": bus_id,
            "theta_rad": theta,
            "theta_deg": theta * 180 / 3.141592653589793,
        }
        for bus_id, theta in result.theta.items()
    ]
)

theta_df

,bus_id,theta_rad,theta_deg
0,1,-0.000000,-0.000000
1,2,-0.117683,-6.742766
2,3,-0.264904,-15.177879
3,4,-0.217948,-12.487494
4,5,-0.187436,-10.739274
5,6,-0.293704,-16.827999
6,7,-0.278451,-15.954077
7,8,-0.278451,-15.954077
8,9,-0.310280,-17.777714
9,10,-0.316124,-18.112544


In [22]:
branch_df = pd.DataFrame(
    [
        {
            "branch_id": branch.id,
            "from_bus": branch.from_bus,
            "to_bus": branch.to_bus,
            "pf_pu": result.branch_flow[branch.id],
            "pf_mw": result.branch_flow[branch.id] * case.base_mva,
            "rate_a_pu": branch.rate_a,
            "rate_a_mw": branch.rate_a * case.base_mva,
            "loading_pct": abs(result.branch_flow[branch.id]) / branch.rate_a * 100 if branch.rate_a > 0 else None,
        }
        for branch in case.branches
        if branch.status == 1
    ]
)

branch_df

,branch_id,from_bus,to_bus,pf_pu,pf_mw,rate_a_pu,rate_a_mw,loading_pct
0,1,1,2,1.796213,179.621289,4.72,472.0,38.055358
1,2,1,5,0.793787,79.378711,1.28,128.0,62.014618
2,3,2,3,0.703988,70.398827,1.45,145.0,48.550915
3,4,2,4,0.512937,51.293659,1.58,158.0,32.464341
4,5,2,5,0.362288,36.228803,1.61,161.0,22.502362
5,6,3,4,-0.238012,-23.801173,1.60,160.0,14.875733
6,7,4,5,-0.658409,-65.840901,6.64,664.0,9.915798
7,8,4,7,0.289323,28.932331,1.41,141.0,20.519384
8,9,4,9,0.166011,16.601055,0.53,53.0,31.322746
9,10,5,6,0.421666,42.166614,1.17,117.0,36.039841
